# 04 · Grounded retrieval with hard exclusions

Guarantee an exclusion such as "no sulfites" by filtering first and ranking second.

| | |
| --- | --- |
| **Needs** | `00_setup.ipynb` in the same folder. The first code cell runs it for you. |
| **LLM** | Optional. With `RUN_LLM = False` (the default in `00_setup.ipynb`) you still see everything retrieval does. |
| **Previous / next** | [`03_hybrid_search.ipynb`](03_hybrid_search.ipynb) / [`05_governed_retrieval.ipynb`](05_governed_retrieval.ipynb) |

*The code mirrors the [`rag-retrieval` journey](https://github.com/instaclustr/labs/tree/main/rag-retrieval) in `instaclustr/labs`. Lines marked "Mirrors" name the file in the journey that each piece comes from. This notebook only searches: it never writes to the cluster.*

### Run the setup

The next cell runs `00_setup.ipynb`. It connects to the cluster, loads the two small models, and defines the helper functions used below. You should see `[PASS]` lines. If you see `[FAIL]`, fix that before going on, using the notes in `00_setup.ipynb`.

In [ ]:
%run ./00_setup.ipynb

### Recap from notebook 02

The grounding phase reuses the BM25 query builder from notebook 02, `bm25_query`. It is repeated here, unchanged, so this notebook runs on its own. Read it again only if you need to: the new part of this notebook starts below.

In [ ]:
def bm25_query(text, k, entities=None, tier=None, exclude=None, require=None, recipe_ids=None):
    """BM25 over name and text (`source` and `entities` are exact-match fields), with optional hard filters and an entity boost."""
    filters, must_not, should = [], [], []
    if recipe_ids:
        filters.append({"terms": {"recipe_id": list(recipe_ids)}})
    if tier:
        filters.append({"term": {"tier": tier}})
    for field, values in (exclude or {}).items():
        must_not.append({"terms": {field: [v.lower() for v in values]}})
    for field, values in (require or {}).items():
        filters += [{"term": {field: v.lower()}} for v in values]
    if entities:
        should.append({"constant_score": {"filter": {"terms": {"entities": entities}}, "boost": 6.0}})
    must = [{"multi_match": {"query": text, "fields": ["recipe_name^4", "entities^3", "text^2", "source"],
                             "type": "best_fields", "operator": "or"}}]
    bool_query = {"must": must, "filter": filters, "must_not": must_not}
    if should:
        bool_query["should"] = should
    return {"size": k, "_source": RECIPE_FIELDS + ["entities"], "query": {"bool": bool_query}}

## Hybrid RAG (NER + rerank)

**The idea.** Split retrieval into two phases:

1. **Ground (BM25).** Keyword search, with **hard filters** applied as real OpenSearch `must_not` and `filter` clauses. The candidate set is *guaranteed* to satisfy the constraints.
2. **Refine (vector).** Re-rank exactly those candidates by meaning. The vector search is restricted to the candidates' IDs, so an excluded recipe can never come back in.

The cell below asks for a fish and chips recipe twice, once without a filter and once with `cautions = sulfites` excluded. No LLM is needed to see the difference.

*Mirrors:* `scripts/hybrid-rag-ner-rerank/src/query.py`.

In [ ]:
NER_RERANK_SYSTEM = (VECTOR_SYSTEM + " Never recommend a recipe that the context marks as containing an excluded allergen.")


def vector_refine(question, recipe_ids, top_k=5, exclude=None, pool=300):
    """kNN restricted to the grounding candidates. `pool` must cover the whole corpus, because the ID filter
    is applied after the nearest-neighbour search."""
    filters = [{"terms": {"recipe_id": list(recipe_ids)}}]
    must_not = [{"terms": {f: [v.lower() for v in vs]}} for f, vs in (exclude or {}).items()]
    body = {"size": top_k, "_source": RECIPE_FIELDS,
            "query": {"bool": {"must": [{"knn": {"embedding": {"vector": embed(question), "k": max(pool, len(recipe_ids))}}}],
                               "filter": filters, "must_not": must_not}}}
    return normalize_hits(search(IDX["vector"], body))


def ground_and_refine(question, top_k=5, candidate_k=25, exclude=None, require=None):
    entities = extract_entities(question)
    grounding = normalize_hits(search(IDX["bm25"], bm25_query(question, candidate_k, entities=entities,
                                                              exclude=exclude, require=require)))
    ids = [h["recipe_id"] for h in grounding if h.get("recipe_id")]
    refined = vector_refine(question, ids, top_k, exclude) if ids else []
    return grounding, refined


def ask_ner_rerank(question, top_k=5, exclude=None, require=None):
    grounding, refined = ground_and_refine(question, top_k, exclude=exclude, require=require)
    context = build_context(refined, cautions=True) or "No recipes matched the requested constraints."
    return answer(f"Question:\n{question}\n\nContext:\n{context}", NER_RERANK_SYSTEM), grounding, refined


QUESTION = "Suggest a British fish and chips recipe."
for label, exclude in [("WITHOUT a filter", None), ("WITH exclude cautions=sulfites", {"cautions": ["sulfites"]})]:
    reply, grounding, refined = ask_ner_rerank(QUESTION, exclude=exclude)
    n_sulfites = sum("sulfites" in [c.lower() for c in cautions_of(h)] for h in refined)
    print(f"\n{label}: {len(grounding)} grounding candidates, {len(refined)} refined hits, "
          f"{n_sulfites} refined hit(s) with a Sulfites caution")
    show_hits(refined)
    print("  ANSWER:", reply)

**Expected:**

- **Without a filter,** some refined hits carry a **Sulfites** caution, and nothing stops the model from suggesting one.
- **With the exclusion,** the count of refined hits with a Sulfites caution is **0** in the final ranking. The exclusion is enforced by the search itself, not by a hope that low scores keep those recipes out.

**What the retriever owns, and what the model does.** The filters decide what reaches the model, and excluded recipes stay excluded. The model then writes an answer from that text, and nothing in the retriever makes the wording correct. Notebook 05 adds a check on that.

**Try this:** change the exclusion to `{"cautions": ["sulfites", "milk"]}`, or add `require={"cuisine_type": ["british"]}`.

### Checkpoint

You should be able to say the difference between a *score* and a *rule*: with the hard exclusion, the count of recipes with a Sulfites caution is 0, because the search itself rejects them.

Next: open [`05_governed_retrieval.ipynb`](05_governed_retrieval.ipynb).